In [ ]:

import json
import os


original_dataset_filename = '/kaggle/input/prepare-dataset/prepare_Dataset.json' 

knowledge_base_filename = 'knowledge_base.json'
validation_set_filename = 'validation_set.json'


if not os.path.exists(original_dataset_filename):
    print("")
else:
    with open(original_dataset_filename, 'r') as f:
        all_data = json.load(f)

    total_size = len(all_data)
    split_point = int(total_size * 0.8)

    knowledge_base_data = all_data[:split_point]
    validation_set_data = all_data[split_point:]

    with open(knowledge_base_filename, 'w') as f:
        json.dump(knowledge_base_data, f, indent=4)

    with open(validation_set_filename, 'w') as f:
        json.dump(validation_set_data, f, indent=4)

In [ ]:
!pip3 install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4 sentence-transformers==2.7.0 chromadb==0.5.0 tqdm --upgrade transformers accelerate bitsandbytes -q

In [ ]:
import json
import torch
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model_name = 'BAAI/bge-large-en-v1.5'
knowledge_base_filename = '/kaggle/working/knowledge_base.json'
validation_set_filename = '/kaggle/working/validation_set.json'
validation_with_embeddings_filename = 'validation_set_with_embeddings.json'
collection_name = "bdd_scenarios_kb"


device = 'cuda' if torch.cuda.is_available() else 'cpu'



embedding_model = SentenceTransformer(embedding_model_name, device=device)

client = chromadb.Client()
.
collection = client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} 
)

In [ ]:
from tqdm.auto import tqdm 

with open(knowledge_base_filename, 'r') as f:
    knowledge_base_data = json.load(f)

ids_to_add = []
records_to_embed = []
metadata_to_add = []

for i, item in enumerate(knowledge_base_data):
    record_key = next((key for key in item if key.startswith('Record')), None)
    feature_key = next((key for key in item if key.startswith('Feature')), None)

    if record_key and feature_key:
        record_text = item[record_key]
        bdd_scenario_text = item[feature_key]

        records_to_embed.append(record_text)
        ids_to_add.append(f"kb_record_{i}")
        metadata_to_add.append({
            "original_record": record_text,
            "bdd_scenario": bdd_scenario_text
        })

embeddings_to_add = embedding_model.encode(
    records_to_embed,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=False
)

collection.add(
    ids=ids_to_add,
    embeddings=embeddings_to_add.tolist(), 
    metadatas=metadata_to_add
)



In [ ]:

from tqdm.auto import tqdm


with open(validation_set_filename, 'r') as f:
    validation_set_data = json.load(f)

validation_records_to_embed = []
enhanced_validation_data = []


for item in validation_set_data:
    record_key = next((key for key in item if key.startswith('Record')), None)
    feature_key = next((key for key in item if key.startswith('Feature')), None)

    if record_key and feature_key:
        record_text = item[record_key]
        bdd_scenario_text = item[feature_key]
        validation_records_to_embed.append(record_text)
        enhanced_validation_data.append({
            "record_text": record_text,
            "golden_scenario": bdd_scenario_text
        })

validation_embeddings = embedding_model.encode(
    validation_records_to_embed,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=False
)

for i, item in enumerate(enhanced_validation_data):
    item['embedding'] = validation_embeddings[i].tolist()

with open(validation_with_embeddings_filename, 'w') as f:
    json.dump(enhanced_validation_data, f, indent=4)


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token= user_secrets.get_secret("HF_demo")

login(token=hf_token)

In [ ]:
import torch
import transformers

model_id = "meta-llama/Llama-3.1-8B"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.float16}, 
    device_map="auto"
)

In [ ]:


!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4 sentence-transformers==2.7.0 chromadb==0.5.0 transformers==4.42.3 accelerate bitsandbytes tqdm evaluate -q


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "Salesforce/codet5p-770m"


tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipeline = pipeline(
    "text2text-generation", 
    model=model, 
    tokenizer=tokenizer
)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "Salesforce/codet5p-220m"


tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipeline = pipeline(
    "text2text-generation", 
    model=model, 
    tokenizer=tokenizer
)


In [ ]:

!pip install -U bitsandbytes

In [ ]:

!pip uninstall -y torch torchvision torchaudio torch_xla

!pip install torch torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

In [ ]:


!pip install torch~=2.3.0 torch_xla~=2.3.0 torchvision~=0.18.0 -f https://storage.googleapis.com/libtpu-releases/index.html


!pip install "transformers>=4.42.0" datasets evaluate sentence-transformers chromadb tqdm -q



In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

model_id = "deepseek-ai/deepseek-coder-33b-base"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config, 
    device_map="auto" 
)


pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id


In [ ]:
!pip install openai -q

In [ ]:
pip install evaluate scikit-learn

In [ ]:

import json
from tqdm.auto import tqdm
import evaluate
import numpy as np
import gc
import torch
import os



def build_prompt(task_record, retrieved_examples):
    prompt = """1.Your role:
You are an expert Quality Assurance engineer specializing in writing Behavior-driven Development (BDD) scenarios in Gherkin syntax.
2.Your task:
Your task is to translate the given "Source Record" into a compliant and accurate “BDD Scenario”. Use the provided examples to understand the correct format and style.
After you have completed the BDD Scenario for the Task, you must stop. Do not generate any further examples or text.
"""
    for i, example in enumerate(retrieved_examples):
        record, scenario = example['original_record'], example['bdd_scenario']
        example_dict = {"Source Record": record, "BDD Scenario": scenario}
        prompt += f"\nExample{i+1}:\n{json.dumps(example_dict, indent=4)}\n"

    task_dict = {"Source Record": task_record}
    task_json_str = json.dumps(task_dict, indent=4).strip()[:-1].strip()
    prompt += f"""
Task:
{task_json_str},
    "BDD Scenario": """
    return prompt

def calculate_metrics(predictions, references, tokenizer):
    print("\n--- Calculating Performance Metrics ---")
    em_score = evaluate.load("exact_match").compute(references=references, predictions=predictions)['exact_match']
    bleu_score = evaluate.load("bleu").compute(predictions=[p if p.strip() else " " for p in predictions], references=[[r] for r in references])['bleu']
    f1_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens, ref_tokens = set(tokenizer.encode(pred)), set(tokenizer.encode(ref))
        common_tokens = pred_tokens.intersection(ref_tokens)
        if not common_tokens: f1_scores.append(0.0); continue
        precision = len(common_tokens) / len(pred_tokens) if pred_tokens else 0
        recall = len(common_tokens) / len(ref_tokens) if ref_tokens else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    f1_score = np.mean(f1_scores)
    return {"f1_score": f1_score, "bleu_score": bleu_score, "exact_match_score": em_score}

with open('validation_set_with_embeddings.json', 'r') as f:
    full_validation_data = json.load(f) 

checkpoint_filename = 'evaluation_results_checkpoint.json'
report_data = []

if os.path.exists(checkpoint_filename):
    with open(checkpoint_filename, 'r') as f:
        report_data = json.load(f)
    start_index = len(report_data)
else:
    start_index = 0

records_to_process = full_validation_data[start_index:]

if records_to_process:
    for i, item in enumerate(tqdm(records_to_process, desc="Evaluating Full Set")):
        
        results = collection.query(query_embeddings=[item['embedding']], n_results=3) 
        final_prompt = build_prompt(item['record_text'], results['metadatas'][0])
        
        outputs = pipeline(final_prompt, max_new_tokens=512, eos_token_id=pipeline.tokenizer.eos_token_id, do_sample=False, pad_token_id=pipeline.tokenizer.eos_token_id)

        response_text = outputs[0]['generated_text']
        generated_scenario_raw= response_text.strip()
        if generated_scenario_raw.startswith('"'): generated_scenario_raw = generated_scenario_raw[1:]
        if generated_scenario_raw.endswith('"}'): generated_scenario_raw = generated_scenario_raw[:-2]
        if generated_scenario_raw.endswith('"'): generated_scenario_raw = generated_scenario_raw[:-1]
        
        try:
            generated_scenario = generated_scenario_raw.encode().decode('unicode_escape')
        except Exception:
            generated_scenario = generated_scenario_raw

        report_data.append({
            "input_record": item['record_text'],
            "expected_scenario": item['golden_scenario'],
            "generated_scenario": generated_scenario
        })

        del results, final_prompt, outputs, response_text
        gc.collect()
        torch.cuda.empty_cache()
        if (i + 1) % 25 == 0 or (i + 1) == len(records_to_process):
            with open(checkpoint_filename, 'w') as f:
                json.dump(report_data, f, indent=4)

predictions = [d['generated_scenario'] for d in report_data]
references = [d['expected_scenario'] for d in report_data]
metrics = calculate_metrics(predictions, references, pipeline.tokenizer)

report_filename = "final_evaluation_report.txt"
report_lines = []

for data in report_data:
    report_lines.append("Input Record:\n")
    report_lines.append(data['input_record'])
    report_lines.append("\n\nExpected BDD scenario:\n")
    report_lines.append(data['expected_scenario'])
    report_lines.append("\n\nGenerated BDD scenario:\n")
    report_lines.append(data['generated_scenario'])
    report_lines.append("\n" + "="*50 + "\n")

report_lines.append("\n. . .\n\n")
report_lines.append(f"F1 Score (Token-based): {metrics['f1_score']:.4f}")
report_lines.append(f"BLEU Score:               {metrics['bleu_score']:.4f}")
report_lines.append(f"Exact Match Score:        {metrics['exact_match_score']:.4f}")

final_report = "\n".join(report_lines)
with open(report_filename, 'w') as f:
    f.write(final_report)


In [ ]:
rm -rf /kaggle/working/*